# Day 0 - Reliable Provider Setup and Diagnostics

**Purpose:** make the environment observable before any agentic-AI practical begins. This notebook checks the active Jupyter kernel, package locations, environment-variable loading, Groq authentication/model discovery, Ollama server/model discovery, and one real inference request from each available provider.

### Why this matters
Agent notebooks often fail for reasons that have nothing to do with LangGraph: the wrong Python kernel, a `.env` file loaded from the wrong directory, a local server reachable from Terminal but not Jupyter, a retired hosted model, or a model that is installed but extremely slow under the chosen generation settings. We diagnose these separately.

> **Run order:** after changing `.env`, installing packages or starting Ollama, restart the kernel and use **Run All** from the top.



>**Dr Julius Sechang Mboli**
>
>**DAIM, University of Hull**
>
>**https://www.hull.ac.uk/staff-directory/julius-mboli**
>
>**https://www.linkedin.com/in/engr-julius-sechang-mboli/**
>
>**https://jsmboli.github.io/**

## Security first - API keys

Never paste a live key into a notebook cell, commit it to Git, or distribute it inside the bootcamp ZIP. Store keys in `resources/.env`, which is deliberately excluded from this distribution.

If a key has ever been pasted into a chat, notebook, screenshot, Git commit or shared ZIP, revoke it and create a replacement.

In [ ]:
from pathlib import Path
import os, sys, json, time, importlib.util
import pandas as pd
pd.set_option('display.max_colwidth', None)

# Locate the package without relying on a fixed working directory.
cwd = Path.cwd().resolve()
RESOURCE_DIR = None
for p in [cwd, *cwd.parents]:
    if (p / "resources" / "src").exists():
        RESOURCE_DIR = p / "resources"
        break
    if (p / "src").exists() and (p / "notebooks").exists() and (p / "data").exists():
        RESOURCE_DIR = p
        break
if RESOURCE_DIR is None:
    raise RuntimeError("Could not locate resources/src. Extract the complete bootcamp ZIP and open this notebook from inside it.")

SRC = RESOURCE_DIR / "src"
DATA = RESOURCE_DIR / "data"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from providers import ProviderRouter, ProviderError, make_messages
from notebook_utils import (
    environment_table, provider_table, response_table,
    run_with_progress, progress_indicator, display_response, display_note,
)

router = ProviderRouter(verbose=True)
display(environment_table(RESOURCE_DIR, router))

## 1. Verify the active kernel and packages

The table above reports the **actual Python executable** running this notebook and the absolute package directories it found. This is the first place to look when Terminal and Jupyter behave differently.

The next cell checks imports and package versions without modifying the environment.

In [3]:
from importlib import metadata

packages = [
    ("python-dotenv", "dotenv"), ("groq", "groq"), ("ollama", "ollama"),
    ("langgraph", "langgraph"), ("langchain-core", "langchain_core"),
    ("langchain-groq", "langchain_groq"), ("langchain-ollama", "langchain_ollama"),
    ("pandas", "pandas"), ("scikit-learn", "sklearn"), ("tqdm", "tqdm"),
]
rows=[]
for dist_name, import_name in packages:
    ok = importlib.util.find_spec(import_name) is not None
    try:
        version = metadata.version(dist_name)
    except metadata.PackageNotFoundError:
        version = None
    rows.append({"package": dist_name, "importable": ok, "version": version})
display(pd.DataFrame(rows))

,package,importable,version
0,python-dotenv,True,1.2.2
1,groq,True,0.37.1
2,ollama,True,0.6.2
3,langgraph,True,1.2.10
4,langchain-core,True,1.5.2
5,langchain-groq,True,1.1.3
6,langchain-ollama,True,1.1.0
7,pandas,True,3.0.5
8,scikit-learn,True,1.9.0
9,tqdm,True,4.67.3


### If packages are missing
Run this **inside the same Jupyter kernel** rather than a random Terminal environment:

```python
%pip install -r ../requirements.txt
```

That path is correct when this notebook is opened from `resources/notebooks/`. If you are working elsewhere, use the absolute `RESOURCE_DIR / "requirements.txt"` value printed above.

## 2. Account and credential setup

### Groq — recommended hosted provider for the live agent/tool demonstrations
1. Go to **https://console.groq.com/** and create/sign in to an account.
2. Open **https://console.groq.com/keys** and create a project API key.
3. Copy `resources/.env.template` to `resources/.env`.
4. Set `GROQ_API_KEY=...` and leave the model as `qwen/qwen3.6-27b` unless instructed otherwise.
5. Restart the notebook kernel.

Groq's official quickstart recommends the `GROQ_API_KEY` environment variable. The key itself is **not** the argument to `os.environ.get`; the argument is the variable name.

### Ollama — no paid API required
1. Install Ollama from **https://ollama.com/download**.
2. Ensure the Ollama application/server is running.
3. In Terminal run `ollama list`.
4. If needed, pull a model such as `ollama pull qwen3.5:0.8b` or `ollama pull llama3.2:latest`.
5. The default endpoint in this pack is `http://127.0.0.1:11434`.

### Optional providers
- OpenAI: **https://platform.openai.com/** → API keys; variable `OPENAI_API_KEY`.
- Gemini: **https://ai.google.dev/gemini-api/docs/api-key**; variable `GEMINI_API_KEY`.
- Anthropic: **https://console.anthropic.com/** → Settings → API keys; variable `ANTHROPIC_API_KEY`.
- DeepSeek: **https://platform.deepseek.com/api_keys**; variable `DEEPSEEK_API_KEY`.

Students do **not** need all providers. Groq + Ollama + the offline mimic route are enough for the core practicals.

## 3. Provider diagnostics

This version deliberately uses the **official Groq Python SDK** for model discovery. A previous low-level `urllib` request could receive Cloudflare error `1010` even while Groq chat completions worked correctly. A diagnostic endpoint must never falsely mark a working provider as unavailable.

The diagnostic also reports Ollama's endpoint, installed models and selected model.

In [4]:
status = run_with_progress("Checking Groq, Ollama and optional providers", router.diagnose, True)
display(provider_table(status))

PRIMARY_PROVIDER = (
    "groq" if status["groq"]["available"]
    else "ollama" if status["ollama"]["available"]
    else "mimic"
)
print("Primary live provider for this notebook:", PRIMARY_PROVIDER)
print("Groq model:", status["groq"].get("model"))
print("Ollama model:", status["ollama"].get("model"))

,provider,available,selected_model,endpoint,diagnostic_s,detail
0,groq,True,qwen/qwen3.6-27b,https://api.groq.com/openai/v1,0.294,Groq SDK authentication succeeded and the configured model is active.
1,ollama,True,qwen3.5:0.8b,http://127.0.0.1:11434,0.192,Ollama reachable; discovered 4 local model(s) using official ollama Python SDK.
2,openai,False,gpt-5.6-luna,,NaN,Key presence check only; Day 0 reports this as optional.
3,gemini,False,gemini-3.7-flash,,NaN,Key presence check only; Day 0 reports this as optional.
4,anthropic,False,claude-sonnet-5,,NaN,Key presence check only; Day 0 reports this as optional.
5,deepseek,False,deepseek-v4-flash,,NaN,Key presence check only; Day 0 reports this as optional.
6,mimic,True,deterministic-classroom-mimic,,NaN,Offline deterministic teaching fallback.


Primary live provider for this notebook: groq
Groq model: qwen/qwen3.6-27b
Ollama model: qwen3.5:0.8b


### Interpreting the result
- **Groq available = True**: SDK authentication succeeded, or a lightweight SDK chat smoke-test succeeded even if model listing did not.
- **Ollama available = True**: the notebook kernel reached the Ollama server and discovered at least one installed model.
- **Mimic available = True**: an offline deterministic fallback is always present so no student is excluded by API cost or account availability.

## 4. Groq smoke test — real inference and metadata

`qwen/qwen3.6-27b` supports both thinking and non-thinking modes. For ordinary classroom calls this pack uses `GROQ_REASONING_EFFORT=none` together with `GROQ_REASONING_FORMAT=hidden` so routine exercises stay fast and do not flood the notebook with long reasoning traces. Set the effort to `default` only when extended reasoning is intentionally part of the exercise. The response panel reports provider, model, latency, token counts, request ID and endpoint where the SDK exposes them.

In [5]:
if not status["groq"]["available"]:
    display_note(status["groq"].get("detail", "Groq unavailable"), "warning")
else:
    groq_response = run_with_progress(
        f"Groq inference — {status['groq']['model']}",
        router.chat,
        make_messages(
            "Give three realistic but bounded agentic-AI use cases for a UK fulfilment centre. "
            "For each, identify one tool and one human-review boundary."
        ),
        provider="groq",
        model=status["groq"]["model"],
        max_tokens=350,
        fallback_on_error=False,
    )
    display_response(groq_response, "Groq smoke-test response")

▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=350
✓ Groq completed in 0.993s | model=qwen/qwen3.6-27b


### Groq smoke-test response

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,groq,qwen/qwen3.6-27b,0.993,64,350,414,length,chatcmpl-46a67763-492a-4ae3-81fd-db869afee59e,https://api.groq.com/openai/v1/chat/completions,format=hidden; effort=none


Here are three realistic, bounded agentic-AI use cases tailored for a UK fulfilment centre, designed to balance automation with necessary human oversight and compliance.

### 1. Dynamic Inventory Rebalancing Agent
**Context:** A fulfilment centre manages thousands of SKUs across multiple zones. Stock levels fluctuate rapidly due to sales velocity, returns, and supplier delays. Manual rebalancing is slow and error-prone.

*   **Agent Role:** The agent monitors real-time inventory levels, sales forecasts, and warehouse zone capacity. It identifies imbalances (e.g., high-demand items stored in distant zones) and generates optimal transfer tasks to move stock to faster-picking locations.
*   **Tool:** **Warehouse Management System (WMS) API Integration** – The agent uses the WMS API to read current stock locations, update task lists, and generate internal transfer orders.
*   **Human-Review Boundary:** **High-Value or Hazardous Goods Exception Handling.**  
    *   *Boundary Rule:* The agent can autonomously generate transfer tasks for standard, low-value, non-hazardous items. However, any proposed transfer involving goods valued over £500, or classified as hazardous (e.g., batteries, chemicals under UK COSHH regulations), must be flagged for human approval before the task is released to pickers. This ensures compliance with safety regulations and reduces risk of loss/theft.

### 2. Automated Returns Triage & Disposition Agent
**Context:** Returns are a major cost centre. Each returned item must be inspected, categorized (resellable, refurbish, recycle, dispose), and processed accordingly. Manual inspection is slow and inconsistent.

*   **Agent Role:**

## 5. Ollama smoke test — local model discovery, warm state and bounded generation

For classroom responsiveness the router defaults to:
- `think=False` (`OLLAMA_THINK=false`),
- a bounded `max_tokens`, and
- `OLLAMA_KEEP_ALIVE=15m` so repeated calls avoid unnecessary reloads.

These defaults are particularly useful for small reasoning models where unrestricted thinking can make a simple classroom prompt take minutes.

In [6]:
ollama_diag = router.diagnose_ollama(refresh=True)
print(ollama_diag.detail)
print("Installed models:", ollama_diag.models)
print("Selected model:", ollama_diag.model)

if not ollama_diag.available:
    display_note(ollama_diag.detail, "warning")
else:
    ollama_response = run_with_progress(
        f"Ollama inference — {ollama_diag.model}",
        router.chat,
        make_messages("Explain chatbot versus agent versus workflow in no more than six concise sentences."),
        provider="ollama",
        model=ollama_diag.model,
        max_tokens=220,
        fallback_on_error=False,
    )
    display_response(ollama_response, "Ollama smoke-test response")

Ollama reachable; discovered 4 local model(s) using official ollama Python SDK.
Installed models: ['qwen3.5:0.8b', 'deepseek-r1:1.5B', 'llama3.2:1b', 'llama3.2:latest']
Selected model: qwen3.5:0.8b


▶ Ollama request started | model=qwen3.5:0.8b | think=False | max_tokens=220 | host=http://127.0.0.1:11434
✓ Ollama completed in 3.014s | model=qwen3.5:0.8b


### Ollama smoke-test response

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,ollama,qwen3.5:0.8b,3.014,50,72,122,stop,None,http://127.0.0.1:11434/api/chat,thinking disabled


A chatbot is designed to answer questions and provide information through natural language interactions without executing tasks autonomously. An agent acts as a specialized tool that can perform specific actions, such as searching data or processing documents, based on explicit instructions provided by the user. A workflow represents an organized sequence of steps executed in order to complete complex objectives within a defined timeframe.

### Optional local runtime check
`ollama ps` shows whether a model is loaded and which processor is being used. It is useful when a local call is unexpectedly slow.

In [8]:
import shutil, subprocess
if shutil.which("ollama"):
    ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
    print(ps.stdout or ps.stderr or "No active Ollama process information returned.")
else:
    print("The Ollama CLI is not on this notebook kernel's PATH. API access may still work through OLLAMA_BASE_URL.")

NAME            ID              SIZE      PROCESSOR    CONTEXT    UNTIL               
qwen3.5:0.8b    f3817196d142    1.6 GB    100% GPU     32768      13 minutes from now    



## 6. Explicit-provider versus automatic routing

For debugging, an explicit provider call **must fail visibly** rather than silently becoming a mimic answer. Automatic routing is different: it may use the next available provider so that a classroom exercise can continue.

In [9]:
probe = "State the five components of a controlled agent workflow in one short list."
rows=[]
for provider in ["groq", "ollama", "mimic"]:
    if provider != "mimic" and not status[provider]["available"]:
        rows.append({"provider": provider, "status": "unavailable", "model": status[provider].get("model")})
        continue
    try:
        r = run_with_progress(
            f"Explicit {provider} probe", router.chat, make_messages(probe),
            provider=provider, max_tokens=180, fallback_on_error=False
        )
        rows.append({**r.summary(), "status": "ok"})
    except ProviderError as exc:
        rows.append({"provider": provider, "status": f"ERROR: {exc}", "model": None})
display(pd.DataFrame(rows))

print("\nAutomatic route:")
auto = run_with_progress("Automatic provider routing", router.chat, make_messages(probe), provider=None, max_tokens=180)
display_response(auto, "Automatic-route response")

▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=180
✓ Groq completed in 0.931s | model=qwen/qwen3.6-27b


▶ Ollama request started | model=qwen3.5:0.8b | think=False | max_tokens=180 | host=http://127.0.0.1:11434
✓ Ollama completed in 1.188s | model=qwen3.5:0.8b


,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode,status
0,groq,qwen/qwen3.6-27b,0.931,48.0,84.0,132.0,stop,chatcmpl-ff3ea5da-05df-4074-bb4b-a210eea84c37,https://api.groq.com/openai/v1/chat/completions,format=hidden; effort=none,ok
1,ollama,qwen3.5:0.8b,1.188,48.0,130.0,178.0,stop,NaN,http://127.0.0.1:11434/api/chat,thinking disabled,ok
2,mimic,deterministic-classroom-mimic,0.000,NaN,NaN,NaN,NaN,NaN,offline,none,ok



Automatic route:


▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=180
✓ Groq completed in 1.190s | model=qwen/qwen3.6-27b


### Automatic-route response

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,groq,qwen/qwen3.6-27b,1.19,48,72,120,stop,chatcmpl-8379a7c2-1ba1-42cc-acb6-dde708d04656,https://api.groq.com/openai/v1/chat/completions,format=hidden; effort=none


1. **Perception** (Observing the environment)
2. **Memory** (Storing and retrieving context)
3. **Planning** (Deciding on a course of action)
4. **Tool Use** (Executing actions via APIs or code)
5. **Reflection** (Evaluating outcomes and self-correction)

## 7. Pre-flight checklist

Before Day 1, confirm that:
- the kernel executable is the environment you intend to use;
- `resources`, `src` and `data` resolve to the extracted bootcamp package;
- Groq works through the SDK if you intend to use hosted inference;
- Ollama reports the models you expect if you intend to use local inference;
- no live key appears in notebook outputs;
- the provider/model/latency metadata panel appears after a successful call.

If Groq chat succeeds but a model-list diagnostic fails, **do not conclude that the API key is bad**. The revised router treats actual chat success as the decisive usability test.